In [12]:
import numpy as np
import torch 
import matplotlib.pyplot as plt
import pandas as pd

In [13]:
df = pd.read_csv('./data/creditcard.csv')

X=torch.from_numpy(df.iloc[:,1:-1].values).float()
y=torch.from_numpy(df.iloc[:,-1].values).long()
X,y

(tensor([[-1.3598e+00, -7.2781e-02,  2.5363e+00,  ...,  1.3356e-01,
          -2.1053e-02,  1.4962e+02],
         [ 1.1919e+00,  2.6615e-01,  1.6648e-01,  ..., -8.9831e-03,
           1.4724e-02,  2.6900e+00],
         [-1.3584e+00, -1.3402e+00,  1.7732e+00,  ..., -5.5353e-02,
          -5.9752e-02,  3.7866e+02],
         ...,
         [ 1.9196e+00, -3.0125e-01, -3.2496e+00,  ...,  4.4548e-03,
          -2.6561e-02,  6.7880e+01],
         [-2.4044e-01,  5.3048e-01,  7.0251e-01,  ...,  1.0882e-01,
           1.0453e-01,  1.0000e+01],
         [-5.3341e-01, -1.8973e-01,  7.0334e-01,  ..., -2.4153e-03,
           1.3649e-02,  2.1700e+02]]),
 tensor([0, 0, 0,  ..., 0, 0, 0]))

In [14]:
X_mean = X.mean(dim=0, keepdim=True)
X_std = X.std(dim=0, keepdim=True)
X = (X - X_mean) / (X_std + 1e-8)

In [15]:
class SoftmaxRegression(torch.nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = torch.nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        # Linear: z = Wx + b
        logits = self.linear(x)
        # Softmax: chuyển thành xác suất
        probs = torch.softmax(logits, dim=1)
        return probs
# Tạo model (2 classes: fraud=1, normal=0)
num_features = X.shape[1]
model = SoftmaxRegression(num_features, 2)
print(model)

SoftmaxRegression(
  (linear): Linear(in_features=29, out_features=2, bias=True)
)


In [16]:
criterion = torch.nn.CrossEntropyLoss()
# Optimizer
learning_rate = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [17]:
num_epochs = 100
losses = []
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(X)
    
    # Tính loss (chú ý: CrossEntropyLoss cần logits, không cần softmax)
    logits = model.linear(X)
    loss = criterion(logits, y)
    
    # Backward và optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.5087
Epoch [20/100], Loss: 0.4317
Epoch [30/100], Loss: 0.3688
Epoch [40/100], Loss: 0.3158
Epoch [50/100], Loss: 0.2736
Epoch [60/100], Loss: 0.2389
Epoch [70/100], Loss: 0.2105
Epoch [80/100], Loss: 0.1869
Epoch [90/100], Loss: 0.1672
Epoch [100/100], Loss: 0.1506


In [18]:
model.eval()
with torch.no_grad():
    logits = model.linear(X)
    predictions = torch.argmax(logits, dim=1)
    accuracy = (predictions == y).sum().item() / len(y)
    print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 99.94%
